In [1]:
import sys
import importlib

def install_if_missing(package, import_name=None):
    """Install a package only if it's not already installed"""
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package}...")
        !{sys.executable} -m pip install {package} -q
        print(f"{package} installed ✅")
    else:
        print(f"{package} already available ✅")

install_if_missing("datasets")
install_if_missing("tqdm")
install_if_missing("ipywidgets")

datasets already available ✅
tqdm already available ✅
ipywidgets already available ✅


In [2]:
import os

import re
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import Adam

from torchvision import transforms
from datasets import load_dataset
from tqdm import tqdm

In [3]:
def add_noise(x0,t, alpha_bars):
    # Compute the noise
    noise = torch.randn_like(x0)
    # Get the wanted alpha bar
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    # Compute xt
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise
    return x_t, noise

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads=4, ff_mult=4):
        super().__init__()

        # Self Attention
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)

        # MLP/Feed Forward
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, dim * ff_mult),
            nn.GELU(),
            nn.Linear(dim * ff_mult, dim)
        )

    def forward(self, x, padding_mask=None):
        # Attention
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, key_padding_mask=padding_mask)
        # Residual
        x = x + attn_out

        # Feed Forward + Residual
        x = x + self.ff(self.norm2(x))
        return x
    

class TextEncoder(nn.Module):
    def __init__(self, vocab_size, dim=128, n_layers=4, n_heads=4, max_len=28):
        """
        vocab_size : size of your tokenizer vocabulary
        dim        : embedding dimension
        n_layers   : number of transformer blocks
        n_heads    : number of attention heads
        max_len    : maximum sequence length
        """
        super().__init__()

        self.dim = dim

        # Embedding (Pos + Token)
        # Each token is projected into a space of dimension dim
        self.token_emb = nn.Embedding(vocab_size, dim, padding_idx=0)
        # Reguarding the position of the token he is also projected into a space of dimension dim
        self.pos_emb = nn.Embedding(max_len, dim)

        # Transformersblocks
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(n_layers)])

        # Norm
        self.norm = nn.LayerNorm(dim)

    def forward(self, tokens, padding_mask=None):
        """
        tokens       : (B, seq_len) token ids
        padding_mask : (B, seq_len) True where padding
        returns      : (B, seq_len, dim) text embeddings
        """
        B, seq_len = tokens.shape

        # Embedding
        # To get the positions of each tokens
        positions = torch.arange(seq_len, device=tokens.device).unsqueeze(0) # (1, seq_len)

        # Compute the embedding
        x = self.token_emb(tokens) + self.pos_emb(positions) # (B, seq_len, dim)

        # Transformers block
        for block in self.blocks:
            x = block(x, padding_mask) # to not influence attetion 

        return self.norm(x) # (B, seq_len, dim)

In [ ]:
class TimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

        # At this point we could implement on top of the sinusoide embedding a small MLP to project better

    def sinusoidal_embedding(self, t):
        half = self.dim // 2
        freqs = torch.exp(-torch.arange(half, device=t.device) * (torch.log(torch.tensor(10000.0)) / (half - 1)))
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return emb

    def forward(self,t):
        return self.sinusoidal_embedding(t)

class ResBlock(nn.Module):
    def __init__(self, in_channels:int,out_channels:int,emb_dim:int):
        """
        in_channels : number of input channels
        out_channels : number of output channels
        emb_dim : dimension of the timestep embedding
        """

        super().__init__()

        # First block
        self.block1 = nn.Sequential(
            nn.GroupNorm(8, in_channels), #split channels into group of 8
            nn.SiLU(),
            nn.Conv2d(in_channels,out_channels, kernel_size=3,padding=1)
        )

        # Embedding projection
        # We need to project the embedding in order to be in the same space while adding
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels)
        )

        # Second block
        self.block2 = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels,out_channels, kernel_size=3,padding=1)
        )

        # Skip connection
        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) #too match if different
        else:
            self.skip = nn.Identity()

    def forward(self,x,t_emb):
        """
        x     : (B, in_channels, H, W)
        t_emb : (B, emb_dim)
        """

        # First block
        # (B, out_channels, H, W)
        h = self.block1(x)

        # Add time step embedding
        # (B, emb_dim)
        t = self.time_proj(t_emb)
        # (B, out_channels)

        # Add by broadcasting
        h = h + t[:, :, None, None]

        # Second block
        # (B, out_channels, H, W)
        h = self.block2(h)

        # Skip connection
        return h + self.skip(x)
    
class SelfAttention(nn.Module):
    def __init__(self, channels:int,n_heads:int=4):
        """
        channels : number of input/output channels
        n_heads  : number of attention heads
        """
        super().__init__()
        assert channels % n_heads == 0

        self.channels = channels
        self.n_heads = n_heads
        self.head_dim = channels // n_heads

        # Normalization
        self.norm = nn.GroupNorm(8, channels)

        # Q,K,V Linear
        self.qkv = nn.Linear(channels, channels * 3)
        self.output = nn.Linear(channels,channels)

    def forward(self,x):
        """
        x : (B, C, H, W)
        """

        B,C,H,W = x.shape

        # Norm
        h = self.norm(x)

        # Flatten
        h = h.view(B, C, H * W).transpose(1, 2) # (B, H*W, C)

        # Compute Q, K, V
        qkv = self.qkv(h) # (B, H*W, 3*C)
        q,k,v = qkv.chunk(3, dim=-1) # 3*(B, H*W, C)

        # At this point, the differents head are inside q,k,v s we need to split thoses matrix depending on n_heads
        def split_heads(t):
            return t.view(B, H * W, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, H*W, head_dim)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Dot-product attention
        scale = self.head_dim ** -0.5
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, n_heads, H*W, H*W)
        attn = torch.softmax(scores, dim=-1) # (B, n_heads, H*W, H*W)

        # Weighted sum of values
        out = torch.matmul(attn, v) # (B, n_heads, H*W, head_dim)

        # Merge heads
        out = out.transpose(1, 2).contiguous() # (B, H*W, n_heads, head_dim)
        out = out.view(B, H * W, C) # (B, H*W, C)

        # Output projection
        out = self.output(out) # (B, H*W, C)

        # Reshape
        out = out.transpose(1, 2).view(B, C, H, W) # (B, C, H, W)

        # Return with residual
        return out + x 
    
class CrossAttention(nn.Module):
    def __init__(self, query_dim, context_dim, n_heads=4):
        """
        query_dim   : dimension of image features (from U-Net)
        context_dim : dimension of text embeddings (from Text Encoder)
        n_heads     : number of attention heads
        """
        super().__init__()
        assert query_dim % n_heads == 0, "query_dim must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = query_dim // n_heads
        self.scale = self.head_dim -0.5

        # Normalization
        self.norm_image = nn.LayerNorm(query_dim)
        self.norm_text = nn.LayerNorm(context_dim)

        # Q from image, K and V from text
        self.q_proj = nn.Linear(query_dim, query_dim)
        self.k_proj = nn.Linear(context_dim, query_dim)
        self.v_proj = nn.Linear(context_dim, query_dim)

        # Output
        self.out_proj = nn.Linear(query_dim, query_dim)

    def forward(self, x, context):
        """
        x       : (B, H*W, query_dim)   image features
        context : (B, seq_len, context_dim) text embeddings
        returns : (B, H*W, query_dim)
        """

        # Get the shape
        B, HW, _ = x.shape
        _, seq_len, _ = context.shape

        # Normalize
        x_norm = self.norm_image(x)
        ctx_norm = self.norm_text(context)

        # Compute Q, K, V 
        q = self.q_proj(x_norm) # (B, H*W, query_dim)

        # projection of context_dim to query_dim
        k = self.k_proj(ctx_norm) # (B, seq_len, query_dim)
        v = self.v_proj(ctx_norm) # (B, seq_len, query_dim)

        # Reshape
        q = q.view(B, HW,self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, HW, head_dim)
        k = k.view(B, seq_len, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, seq_len, head_dim)
        v = v.view(B, seq_len, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, seq_len, head_dim)

        # Scaled dot-product attention
        out = F.scaled_dot_product_attention(q, k, v) # (B, n_heads, HW, head_dim)

        # Merge heads
        out = out.transpose(1, 2).contiguous().view(B, HW, -1) # (B, HW, query_dim)

        # Output + residual
        return self.out_proj(out) + x # (B, H*W, query_dim)
        

class UNet(nn.Module):
    def __init__(
        self, 
        in_channels:int=3,
        base_channels:int=32,
        channel_mults = (1,2,4),
        n_heads = 4,
        emb_dim =128,
        context_dim=128,
    ):
        super().__init__()

        self.emb_dim = emb_dim
        channels = [base_channels * i for i in channel_mults]

        # Timestep embedding
        self.time_emb = TimestepEmbedding(emb_dim)

        # init conv
        self.init_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)


        ###########
        # Encoder #
        ###########

        # To increase the number of channels and learn multiple features
        self.encoder_blocks = nn.ModuleList()
        # To reduce the size of the input and see the features more precisely/largelly
        self.downsamplers = nn.ModuleList()

        in_ch = base_channels
        for out_ch in channels:
            self.encoder_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch, emb_dim), # Changes the channel dimension
                ResBlock(out_ch, out_ch, emb_dim), # Work in the same space -> extracts more complex patterns
            ]))
            self.downsamplers.append(
                nn.Conv2d(out_ch, out_ch, kernel_size=4, stride=2, padding=1)
            )
            in_ch = out_ch

        ##############
        # Bottleneck #
        ##############
        mid_ch = channels[-1]

        bottleneck_blocks = [
            ResBlock(mid_ch, mid_ch, emb_dim),
            SelfAttention(mid_ch, n_heads),
        ]

        if context_dim is not None:
            bottleneck_blocks.append(CrossAttention(mid_ch, context_dim))

        bottleneck_blocks.append(ResBlock(mid_ch, mid_ch, emb_dim))

        self.bottleneck = nn.ModuleList(bottleneck_blocks)

        ###########
        # Decoder #
        ###########
        # Reduce the number of channels and features
        self.decoder_blocks = nn.ModuleList()
        # Increase the size of the value inside the bottlenck
        self.upsamplers = nn.ModuleList()

        for out_ch in reversed(channels):
            self.upsamplers.append(
                nn.ConvTranspose2d(in_ch, in_ch, kernel_size=4, stride=2, padding=1)
            )
            self.decoder_blocks.append(nn.ModuleList([
                ResBlock(in_ch + out_ch, out_ch, emb_dim), # +out_ch for skip connection
                ResBlock(out_ch, out_ch, emb_dim),
            ]))
            in_ch = out_ch

        # Final conv that predict the noise
        self.final = nn.Sequential(
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1)
        )
        
    def forward(self, x, t, text_emb=None):
        """
        x : (B, 3, 32, 32) noisy image
        t : (B,)            timestep
        text_emb : (B, seq_len, context_dim)
        returns predicted noise ε (B, 3, 32, 32)
        """

        # Timestep embedding
        t_emb = self.time_emb(t) # (B, emb_dim)

        x = self.init_conv(x) 

        ### Encoder ###
        skips = []
        for (res1, res2), down in zip(self.encoder_blocks, self.downsamplers):
            x = res1(x, t_emb) # First ResBlock
            x = res2(x, t_emb) # Second ResBlock
            skips.append(x)
            x = down(x) # DownSample

        ### Bottleneck ###
        B, C, H, W = x.shape
        
        if text_emb is not None:
            res1, attn, cross, res2 = self.bottleneck
        else:
            res1, attn, res2 = self.bottleneck

        # Classic
        x = res1(x, t_emb)
        x = attn(x)

        # If text conditionning
        if text_emb is not None:
            x_flat = x.view(B, C, H * W).transpose(1, 2) # (B, H*W, C)
            x_flat = cross(x_flat, text_emb) # (B, H*W, C)
            x = x_flat.transpose(1, 2).view(B, C, H, W) # (B, C, H, W)

        # Classic
        x = res2(x, t_emb)

        ### Decoder ###
        for (res1, res2), up, skip in zip(self.decoder_blocks, self.upsamplers, reversed(skips)):
            x = up(x) # UpSample
            x = torch.cat([x, skip], dim=1) # Add the residual
            x = res1(x, t_emb) # First ResBlock
            x = res2(x, t_emb) # Second ResBlock

        # Final conv
        return self.final(x)

In [ ]:
def loadPixelArtDataset():
    return load_dataset("jiovine/pixel-art-nouns")['train']

class PixelArtDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, image_size: int = 32):
        self.dataset = hf_dataset
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        img = sample['image'].convert('RGB')
        img = self.transform(img)
        text = sample['text']
        return {'image': img, 'text': text}
    
def displayRandomSample(dataset):
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    axes = axes.flatten()

    indices = random.sample(range(len(dataset)), 10)

    for i, ax in enumerate(axes):
        sample = dataset[indices[i]]
        img = sample['image']
        
        ax.imshow(img, interpolation='nearest')
        ax.axis('off')

    plt.suptitle("PixelDiffusion — Random samples", fontsize=13)
    plt.tight_layout()
    plt.show()

In [7]:
def tokenize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s-]', '', text)
    text = text.replace('-', ' - ')
    return text.split()

def tokenize_dataset(texts):
    word_counts = Counter()
    for text in texts:
        word_counts.update(tokenize_text(text))

    special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>']

    vocab = special_tokens + [word for word, count in word_counts.most_common()]

    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    
    return vocab, word2idx, idx2word
    
def encode(text, max_len,word2idx):
    tokens = tokenize_text(text)
    ids = [word2idx['<BOS>']]
    ids += [word2idx.get(t, word2idx['<UNK>']) for t in tokens]
    ids += [word2idx['<EOS>']]
    # Pad or truncate
    ids = ids[:max_len]
    ids += [word2idx['<PAD>']] * (max_len - len(ids))
    return ids

def decode(ids,word2idx,idx2word):
    words = [idx2word[i] for i in ids 
             if i not in [word2idx['<PAD>'], word2idx['<BOS>'], word2idx['<EOS>']]]
    return ' '.join(words)

def plot_token_info(texts):
    token_lengths = [len(tokenize_text(text)) for text in texts]
    avg_len = sum(token_lengths) / len(token_lengths)

    # Histogram
    plt.figure(figsize=(10, 4))
    plt.hist(token_lengths, bins=20, color='steelblue', edgecolor='white')
    plt.axvline(avg_len, color='red', linestyle='--', label=f'Average ({avg_len:.1f})')
    plt.axvline(max(token_lengths), color='orange', linestyle='--', label=f'Max ({max(token_lengths)})')
    plt.xlabel("Number of tokens")
    plt.ylabel("Number of samples")
    plt.title("PixelDiffusion — Token length distribution")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from torch.amp import autocast, GradScaler

class Trainer:
    def __init__(self, config: dict):
        """
        config keys:
            device          : torch.device
            checkpoint_dir  : str
            epochs          : int
            T               : int
            batch_size      : int
            image_size      : int
            dropout_cond   : float (0.0 = no CFG)
        """
        self.config = config
        self.device = config["device"]
        self.checkpoint_dir = config.get("checkpoint_dir", "./checkpoints")
        self.epochs = config["epochs"]
        self.T = config["T"]
        self.dropout_cond = config.get("dropout_cond", 0.0)
        self.start_epoch = 0
        self.losses = []
        self.scaler = GradScaler()

        os.makedirs(self.checkpoint_dir, exist_ok=True)
        
        
    def train(self,unet:UNet,optimizer,text_encoder:TextEncoder,train_loader,word2idx,max_len=30,pretrained:str=None,conditionned:bool=False,tag:str=""):
        
        # Load pretrained weight
        if pretrained != None:
            
            self._load_weight(unet, optimizer, text_encoder, pretrained)
        
        # Init some values for the noise
        beta_start = 1e-4
        beta_end = 0.02

        betas = torch.linspace(beta_start, beta_end, self.T).to(self.device)
        alphas = (1 - betas).to(self.device)
        alpha_bars = torch.cumprod(alphas, dim=0).to(self.device)
        
        # Training loop
        for epoch in range(self.start_epoch,self.epochs):
            
            # Init
            epoch_loss = 0.0
            
            # Put the model into train mode
            unet.train()
            if text_encoder is not None : 
                text_encoder.train()
                
            # Tqdm bar
            batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{self.epochs}", leave=False)
            
            for batch in batch_bar:
                
                # Get the image
                x0 = batch['image'].to(self.device) # (B, 3, 32, 32)
                B = x0.shape[0]
                
                # Text embedding part
                text_emb = None
                if text_encoder is not None:
                    # Get the text
                    texts = batch['text']

                    # Tokenized the text
                    tokens = torch.tensor([encode(t,max_len,word2idx) for t in texts], dtype=torch.long).to(self.device) # (B, seq_len)
                    padding_mask = (tokens == word2idx['<PAD>']).to(self.device) # (B, seq_len)
                    text_emb = text_encoder(tokens, padding_mask) # (B, seq_len, 128)
                    
                # Noise
                timesteps = torch.randint(0, self.T, (B,), device=self.device)
                x_t, noise = add_noise(x0, timesteps,alpha_bars) # (B, 3, 32, 32)
                
                # Forward + loss
                with autocast('cuda'):
                    noise_pred = unet(x_t, timesteps, text_emb)
                    loss = F.mse_loss(noise_pred, noise)

                # Backprop
                optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.step(optimizer)
                self.scaler.update()

                # Update some values
                epoch_loss += loss.item()
                batch_bar.set_postfix(loss=f"{loss.item():.6f}")

            avg_loss = epoch_loss / len(train_loader)
            self.losses.append(avg_loss)
            
            if epoch % 10 == 0:
                self._save(epoch,unet,optimizer,text_encoder,avg_loss,self.losses,tag)
                print(f"Epoch {epoch:4d} / {self.epochs} | Avg Loss : {avg_loss:.6f} | Saved")
                
    def _save(self, epoch:int, unet:UNet, optimizer, text_encoder:TextEncoder, avg_loss:float, losses, tag: str):
        """Save a checkpoint every 10 epochs."""
        if epoch % 10 != 0:
            return

        raw_model = unet.module if isinstance(unet, nn.DataParallel) else unet
        payload = {
            "epoch" : epoch,
            "unet_state" : raw_model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "loss" : avg_loss,
            "losses" : losses,
        }

        if text_encoder is not None:
            raw_enc = text_encoder.module if isinstance(text_encoder, nn.DataParallel) else text_encoder
            payload["text_encoder_state"] = raw_enc.state_dict()

        filename = f"pixeldiffusion_{tag}_epoch_{epoch}.pt"
        torch.save(payload, os.path.join(self.checkpoint_dir, filename))
        
    def _load_weight(self, unet: UNet, optimizer, text_encoder: TextEncoder, pretrained: str):
        full_path = os.path.join(self.checkpoint_dir, pretrained)
        try:
            ckpt = torch.load(full_path, map_location=self.device)

            # Load UNet 
            raw_unet = unet.module if isinstance(unet, nn.DataParallel) else unet
            state_dict = ckpt["unet_state"]

            # Detect if checkpoint is unconditional (no CrossAttention key saved)
            is_uncond_checkpoint = not any("bottleneck.2" in k and "cross" in k.lower() for k in state_dict.keys())

            if is_uncond_checkpoint:
                # Remap
                remapped = {
                    (k.replace("bottleneck.2", "bottleneck.3") if k.startswith("bottleneck.2") else k): v
                    for k, v in state_dict.items()
                }
            else:
                # Cond checkpoint : keys already match
                remapped = state_dict

            raw_unet.load_state_dict(remapped, strict=False)

            # Load optimizer
            if "optimizer_state" in ckpt:
                saved_groups = ckpt["optimizer_state"].get("param_groups", [])
                if len(saved_groups) == len(optimizer.param_groups):
                    optimizer.load_state_dict(ckpt["optimizer_state"])
                else:
                    print(f"Optimizer param group mismatch ({len(saved_groups)} vs {len(optimizer.param_groups)}), skipping optimizer state")

            # Load TextEncoder
            if text_encoder is not None and "text_encoder_state" in ckpt:
                raw_enc = text_encoder.module if isinstance(text_encoder, nn.DataParallel) else text_encoder
                raw_enc.load_state_dict(ckpt["text_encoder_state"], strict=False)

            self.start_epoch = ckpt.get("epoch", 0) + 1
            self.losses = ckpt.get("losses", [])
            print(f"Resuming from epoch {self.start_epoch} ({full_path})")

        except FileNotFoundError:
            print(f"No checkpoint found at {full_path} — training from scratch")
        
        

In [25]:
config = {
    "training": {
        "epochs" : 150,
        "batch_size" : 254,
        "T" : 1000,
        "checkpoint_dir": "./",
    },
    "model": {
        "image_size" : 64,
        "in_channels" : 3,
        "base_channels": 32,
        "channel_mults": [1, 2, 4,8],
        "n_heads" : 4,
        "emb_dim" : 128,
        "context_dim" : 128,
        "lr":5.0e-4,
    },
    "text_encoder": {
        "max_len": 30,
        "n_layers": 4,
        "lr":2.0e-4,
    },
}

In [ ]:
# Load the config
cfg = config
train_cfg = cfg["training"]
model_cfg = cfg["model"]
text_encoder_cfg = cfg["text_encoder"]

# Load the dataset
dataset = loadPixelArtDataset()

# Create the data loader
train_dataset = PixelArtDataset(dataset,model_cfg["image_size"])
train_loader = DataLoader(train_dataset, 
                          batch_size=train_cfg["batch_size"],
                          num_workers = 0,
                          pin_memory = True,
                          shuffle=True)

# Display random images of the dataset
#displayRandomSample(dataset)

# Print informations
print(f"Dataset size : {len(train_dataset)}")
print(f"Batches per epoch : {len(train_loader)}")

# Get word2ix and vocab_size 
vocab, word2idx, _ = tokenize_dataset(dataset['text'])

# Check the device used
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device used : ", device)

# Create the unet model
unet = UNet(
    in_channels = model_cfg["in_channels"],
    base_channels = model_cfg["base_channels"],
    channel_mults = model_cfg["channel_mults"],
    n_heads = model_cfg["n_heads"],
    emb_dim = model_cfg["emb_dim"],
    context_dim = model_cfg["context_dim"],
).to(device)

# Create the text encoder model
text_encoder = TextEncoder(len(word2idx),
 dim=model_cfg["context_dim"],
 n_layers=text_encoder_cfg["n_layers"], 
 n_heads=model_cfg["n_heads"],
 max_len=text_encoder_cfg["max_len"]).to(device)

# Display the number of parameters
trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"Trainable parameters in Unet : {trainable}")

trainable = sum(p.numel() for p in text_encoder.parameters() if p.requires_grad)
print(f"Trainable parameters in the Text Encoder : {trainable}")

# Multi-GPU if available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    unet = torch.nn.DataParallel(unet)
    
# Create the optimizer
optimizer = Adam([
    {'params': unet.parameters(),'lr': model_cfg["lr"]},
    {'params': text_encoder.parameters(), 'lr': text_encoder_cfg["lr"]},
])

# Pretrained weight
pretrained = "../checkpoints/pixeldiffusion__epoch_90.pt"

# Create the trainer
trainer = Trainer({
"device" : device,
"checkpoint_dir" : train_cfg["checkpoint_dir"],
"epochs" : train_cfg["epochs"],
"T" : train_cfg["T"],
 "cfg_drop_prob" : 0,})

# Train the model
trainer.train(unet = unet, optimizer = optimizer, text_encoder = text_encoder, train_loader = train_loader, word2idx = word2idx, max_len = text_encoder_cfg["max_len"], pretrained = pretrained, tag="conditionned")

Dataset size : 49859
Batches per epoch : 197
Device used :  cpu
Trainable parameters in Unet : 14067363
Trainable parameters in the Text Encoder : 837632
Optimizer param group mismatch (1 vs 2), skipping optimizer state
Resuming from epoch 91 (./../checkpoints/pixeldiffusion__epoch_90.pt)


KeyboardInterrupt: 